# Lesson 4: A first language model — the bigram model

Now the model makes predictions. A bigram model uses the current character to score the next character. Its entire parameter table is a vocabulary-by-vocabulary lookup table. We will inspect those scores, measure them with cross-entropy, and sample a short continuation from an **untrained** model. The original notebook introduces this in cell 13; training comes in a later lesson so we can first see what the model does before learning.

The matching [C entry point](../c/src/bigram_main.c) delegates the model math to [bigram_model.c](../c/src/bigram_model.c). The [Java lesson](../java/src/main/java/learning/karpathy/gpt1/BigramLanguageModelLesson.java) uses [BigramLanguageModel](../java/src/main/java/learning/karpathy/gpt1/BigramLanguageModel.java). They use ordinary arrays to make the table and shapes visible.

## Prepare tokens and a small batch

This setup repeats the earlier lessons so this notebook opens independently. The batch has four training windows, each eight characters long. Each target is the character immediately after its corresponding input.

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

data_candidates = (
    Path.cwd() / "data" / "input.txt",
    Path.cwd() / "gpt-1" / "data" / "input.txt",
    Path.cwd().parent / "data" / "input.txt",
)
dataset_path = next((path.resolve() for path in data_candidates if path.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError("Could not find gpt-1/data/input.txt. Open the repo root, gpt-1/, or gpt-1/python/.")

text = dataset_path.read_text(encoding="utf-8")
chars = sorted(set(text))
stoi = {character: token_id for token_id, character in enumerate(chars)}
itos = {token_id: character for token_id, character in enumerate(chars)}
encode = lambda sample: [stoi[character] for character in sample]
decode = lambda token_ids: "".join(itos[token_id] for token_id in token_ids)
data = torch.tensor(encode(text), dtype=torch.long)

split_index = int(0.9 * len(data))
train_data = data[:split_index]
val_data = data[split_index:]
batch_size = 4
block_size = 8
torch.manual_seed(1337)

def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x_batch = torch.stack([source[start:start + block_size] for start in starts])
    y_batch = torch.stack([source[start + 1:start + block_size + 1] for start in starts])
    return x_batch, y_batch

x_batch, y_batch = get_batch("train")
print(f"Dataset: {dataset_path}")
print(f"Vocabulary: {len(chars)} characters; train/validation: {len(train_data):,}/{len(val_data):,} tokens")
print(f"Batch shapes: x={tuple(x_batch.shape)}, y={tuple(y_batch.shape)}")

## Step 1: Look up scores for the next character

`nn.Embedding(V, V)` is a learned table with one row for every possible input character and one column for every possible next character. Looking up an input token selects its row. These values are called **logits**: scores that are not probabilities yet. For an input batch shaped `(B, T)`, the output has shape `(B, T, V)`.

This first model is deliberately simple: at each position it looks only at the character at that position. It does not use the earlier characters in the context window.

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Each input-token ID selects one row of next-token scores.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx: (B, T); logits: (B, T, V)
        logits = self.token_embedding_table(idx)
        loss = None
        if targets is not None:
            batch, time, vocab = logits.shape
            # Cross-entropy expects rows of class scores and one target ID per row.
            loss = F.cross_entropy(logits.reshape(batch * time, vocab), targets.reshape(batch * time))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # Repeatedly sample one next ID and append it to the sequence.
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            last_position_scores = logits[:, -1, :]  # (B, V)
            probabilities = F.softmax(last_position_scores, dim=-1)
            next_id = torch.multinomial(probabilities, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

model = BigramLanguageModel(len(chars))
logits, _ = model(x_batch)
print(f"Parameter table shape: {tuple(model.token_embedding_table.weight.shape)}")
print(f"Input shape: {tuple(x_batch.shape)} -> logits shape: {tuple(logits.shape)}")
print(f"First input token ID: {x_batch[0, 0].item()}")
print(f"Its first 8 next-character logits: {logits[0, 0, :8].detach().tolist()}")

## Step 2: Measure prediction error with cross-entropy

At each position, the model must assign high probability to the correct next character. Softmax turns the logits into probabilities. Cross-entropy takes the negative log probability of the target and averages this score over all `B × T` positions. A lower loss means the correct next characters received more probability.

PyTorch's `F.cross_entropy` accepts logits directly; it combines the numerically stable softmax and negative-log calculation. The manual one-position calculation below makes the meaning concrete.

In [ ]:
_, loss = model(x_batch, y_batch)
first_target_id = y_batch[0, 0].item()
first_position_probabilities = F.softmax(logits[0, 0], dim=-1)
first_target_probability = first_position_probabilities[first_target_id].item()
manual_one_position_loss = -torch.log(first_position_probabilities[first_target_id]).item()

print(f"First input character: {decode([x_batch[0, 0].item()])!r}")
print(f"Correct next character: {decode([first_target_id])!r} (ID {first_target_id})")
print(f"Model probability for that target: {first_target_probability:.4f}")
print(f"One-position negative log probability: {manual_one_position_loss:.4f}")
print(f"Mean cross-entropy across {batch_size * block_size} targets: {loss.item():.4f}")
assert torch.isfinite(loss) and loss.item() > 0

## Step 3: Generate a continuation by sampling

Generation starts with one token. At each iteration we take the last position's logits, apply softmax, randomly sample one token from that distribution, and append it. The new token becomes the context for the next iteration. Since this model has not been trained, its continuation is random-looking. The important point here is the autoregressive loop.

In [ ]:
torch.manual_seed(1337)
start = torch.zeros((1, 1), dtype=torch.long)  # token ID 0, often a newline in this vocabulary
sampled_ids = model.generate(start, max_new_tokens=100)[0].tolist()
print(f"Start token: {start.item()} ({decode([start.item()])!r})")
print(decode(sampled_ids))
assert len(sampled_ids) == 101

## Compare and experiment

Run the equivalent C or Java steps from **Terminal → Run Task → Run tutorial lesson**, then choose lesson `4`. The C model stores its table and logits in flat row-major `float` arrays; Java uses `float[][]` and `float[][][]`. Those are the raw array representations of a two-dimensional parameter table and a three-dimensional batch of logits. PyTorch's `nn.Embedding` and tensors package these operations.

Try changing the start token, batch size, or number of generated characters. Why can this model not use the full eight-character context window? In the next training stage, the loss will be used to update the table.

References: [PyTorch `nn.Embedding`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html), [PyTorch cross-entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html), [PyTorch multinomial sampling](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html), and the original [GPT tutorial notebook](../gpt_dev.ipynb), cell 13.